# Etapa 2b — Signo de las componentes principales

En `02_mapas_color.ipynb` las componentes principales se calculan en cada imagen y su signo se fija con una regla que no tiene que ver con los vasos (el coeficiente de mayor valor absoluto, positivo). En 08_dr la segunda componente salió invertida (área 0,19). La tercera, elegida como mapa principal, no se invirtió en desarrollo, pero nada lo garantiza en imágenes nuevas.

**Reglas que se comparan** (ninguna usa las máscaras de los expertos):

| Regla | Cómo se fija el signo |
|---|---|
| A. Coeficiente mayor (la de la etapa 2) | componentes por imagen; el coeficiente de mayor valor absoluto, positivo |
| B. Oscuridad local | componentes por imagen; cada una se orienta para correlacionar positivamente con "mediana local del verde − verde" (`sign_by_vessels`): los vasos quedan como valores altos |
| C. Direcciones fijas | componentes calculadas una sola vez con píxeles de las 30 imágenes de desarrollo (100.000 por imagen, campo detectado) y aplicadas igual a todas: el signo no puede cambiar entre imágenes |

**Qué se mira:**
1. En desarrollo, el área bajo la curva de la segunda y la tercera componente, con un sentido por componente elegido por la media (como en la etapa 2): media, peor imagen y cuántas imágenes quedan por debajo de 0,5 (invertidas).
2. Estabilidad del signo ante cambios de la foto (exposición × 0,5 y × 1,6, .jpg calidad 20): en cuántos casos la componente de la foto alterada queda con el signo contrario al de la original.

**Criterio:** se prefiere la regla sin imágenes invertidas y con signo estable; si hay más de una, la de mayor área media en la tercera componente, y ante empate (diferencia menor que 0,02) la que se calcula en cada imagen, porque se adapta mejor a otra cámara.


In [1]:
import io
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from PIL import Image

from src.color import pca_directions, pca_maps, sign_by_max_coef, sign_by_vessels
from src.config import SEED
from src.data import dev_ids, load_fov, load_image, load_vessels
from src.fov import detect_fov_robust
from src.metrics import auc

ids = dev_ids()
rng = np.random.default_rng(SEED)

## Direcciones fijas (regla C): una sola vez con desarrollo

In [2]:
samples = []
for image_id in ids:
    rgb = load_image(image_id)
    px = rgb[detect_fov_robust(rgb)]
    samples.append(px[rng.choice(len(px), 100_000, replace=False)])
pool = np.concatenate(samples).astype(np.float32) / 255.0
g_mean = pool.mean(axis=0, dtype=np.float64)
g_val, g_vec = np.linalg.eigh(np.cov(pool, rowvar=False))
order = np.argsort(g_val)[::-1]
g_val, g_vec = g_val[order], sign_by_max_coef(g_vec[:, order])
print("Variación explicada:", np.round(100 * g_val / g_val.sum(), 2))
print("Direcciones (columnas PC1, PC2, PC3; filas rojo, verde, azul):")
print(np.round(g_vec, 3))
del pool, samples

Variación explicada: [85.12 12.52  2.36]
Direcciones (columnas PC1, PC2, PC3; filas rojo, verde, azul):
[[ 0.876 -0.476 -0.073]
 [ 0.418  0.827 -0.377]
 [ 0.24   0.3    0.923]]


## Área bajo la curva por regla, componente e imagen

In [3]:
def components(rgb, region, rule):
    if rule == "C":
        return pca_maps(rgb, g_mean, g_vec)
    mean, vec, _ = pca_directions(rgb, region)
    vec = sign_by_max_coef(vec) if rule == "A" else sign_by_vessels(rgb, region, mean, vec)
    return pca_maps(rgb, mean, vec)

rows = []
for image_id in ids:
    rgb, fov, ves = load_image(image_id), load_fov(image_id), load_vessels(image_id)
    region = detect_fov_robust(rgb)
    for rule in "ABC":
        maps = components(rgb, region, rule)
        for name in ("PC2", "PC3"):
            rows.append({"image": image_id, "rule": rule, "component": name,
                         "auc_bright": auc(maps[name], ves, fov)})
res = pd.DataFrame(rows)
res.to_csv(ROOT / "results" / "02b_signo.csv", index=False)

def summarize(d):
    bright = d["auc_bright"]
    oriented = bright if bright.mean() >= 0.5 else 1 - bright
    return pd.Series({"sentido": "claros" if bright.mean() >= 0.5 else "oscuros",
                      "media": oriented.mean(), "peor": oriented.min(),
                      "invertidas (< 0,5)": int((oriented < 0.5).sum())})

summary = res.groupby(["component", "rule"]).apply(summarize, include_groups=False)
summary

sentido     media      peor  invertidas (< 0,5)
component rule                                                 
PC2       A     oscuros  0.758240  0.194549                   1
          B      claros  0.778603  0.682871                   0
          C     oscuros  0.777450  0.691312                   0
PC3       A      claros  0.811491  0.716027                   0
          B      claros  0.811491  0.716027                   0
          C      claros  0.757004  0.598655                   0

## Estabilidad del signo ante cambios de la foto (reglas A y B)

Para cada imagen y cada cambio se calcula la componente con la foto alterada y se correlaciona con la de la foto original (a 1/4 de resolución, dentro del campo detectado). Una correlación negativa quiere decir que el signo se invirtió. La regla C es una proyección fija y no puede invertirse.

In [4]:
def jpeg(a, q):
    buf = io.BytesIO()
    Image.fromarray(a).save(buf, format="JPEG", quality=q)
    return np.asarray(Image.open(buf))

VARIANTS = {"exposición × 0,5": lambda a: (a * 0.5).astype(np.uint8),
            "exposición × 1,6": lambda a: (a * 1.6).clip(0, 255).astype(np.uint8),
            ".jpg calidad 20": lambda a: jpeg(a, 20)}

flip_rows = []
for image_id in ids:
    rgb = load_image(image_id)
    region = detect_fov_robust(rgb)
    small = region[::4, ::4]
    for rule in "AB":
        base = components(rgb, region, rule)
        for vname, f in VARIANTS.items():
            alt = f(rgb)
            other = components(alt, detect_fov_robust(alt), rule)
            for name in ("PC2", "PC3"):
                r = np.corrcoef(base[name][::4, ::4][small], other[name][::4, ::4][small])[0, 1]
                flip_rows.append({"image": image_id, "rule": rule, "variant": vname,
                                  "component": name, "corr": r})
flips = pd.DataFrame(flip_rows)
flips.groupby(["component", "rule", "variant"])["corr"].agg(
    invertidas=lambda c: int((c < 0).sum()), peor="min").round(3)

invertidas   peor
component rule variant                            
PC2       A    .jpg calidad 20            0  0.890
               exposición × 0,5           0  0.997
               exposición × 1,6          25 -0.866
          B    .jpg calidad 20            0  0.890
               exposición × 0,5           0  0.997
               exposición × 1,6           6 -0.560
PC3       A    .jpg calidad 20            0  0.311
               exposición × 0,5           0  0.978
               exposición × 1,6           1 -0.286
          B    .jpg calidad 20            0  0.311
               exposición × 0,5           0  0.978
               exposición × 1,6           0  0.286

## Decisión (escrita después de correr)

**Se adopta la regla B, oscuridad local** (`PCA_SIGN_RULE = "vessels"` en `config.py`).

| | A (etapa 2) | B (oscuridad local) | C (direcciones fijas) |
|---|---|---|---|
| Tercera componente: media / peor / invertidas | 0,811 / 0,716 / 0 | **0,811 / 0,716 / 0** | 0,757 / 0,599 / 0 |
| Segunda componente: media / peor / invertidas | 0,758 / 0,195 / **1** (08_dr) | **0,779 / 0,683 / 0** | 0,777 / 0,691 / 0 |
| Tercera componente invertida al sobreexponer × 1,6 | 1 de 30 | **0 de 30** | no puede invertirse |
| Segunda componente invertida al sobreexponer × 1,6 | **25 de 30** | 6 de 30 | no puede invertirse |

- **En desarrollo, A y B dan lo mismo en la tercera componente** en las 30 imágenes: los resultados de la etapa 2 no cambian. B corrige la segunda componente de 08_dr.
- **B protege a las imágenes nuevas:** con la foto sobreexpuesta, A invierte la tercera componente en una imagen y la segunda en 25; B no invierte la tercera en ninguna.
- **C** nunca se invierte, pero su tercera componente separa peor (0,757 contra 0,811; más de 0,02 de diferencia): las direcciones calculadas en cada imagen se adaptan mejor que unas fijas.
- Con la compresión fuerte y la subexposición ninguna regla invierte el signo.

**Limitación que queda.** Al sobreexponer (el rojo se satura), la tercera componente de la foto alterada se parece poco a la original (correlación de hasta 0,29), aunque no se invierta: la saturación cambia el reparto del color y la componente pasa a representar otra cosa. Es un punto para la etapa de robustez (8).
